# DuctSense: Binary Acoustic Leak Detection Training Pipeline

**Target Model:** Binary Classifier (Random Forest)  
- `NO_LEAK = 0`  
- `LEAK = 1`

### Four Target Operational Scenarios:
1. **No Leak + No Noise** -> `NO_LEAK`
2. **No Leak + Noise** -> `NO_LEAK`
3. **Leak + No Noise** -> `LEAK`
4. **Leak + Noise** -> `LEAK`

### Critical Validation & Data Integrity Rules:
- **IICA Dataset (`TUBELEAK`, `VENTLEAK`, `VENTLOW`):** All recordings across acoustic conditions (`lab`, `hydr`, `hydr_low`, `work`, `work_low`) are leak recordings under varying noise/operating conditions. They are **never** labeled as `NO_LEAK`.
- **Recording-Level Split:** The IICA recordings are split at the WAV file level *before* any windowing. Windows from the same recording are strictly prohibited from appearing in both train and test partitions.
- **NO_LEAK Environmental Audio:** Built from continuous physical HVAC recordings (`RoomTone2` and `Empty Mall`). Windows are partitioned temporally (first 80% train, last 20% test).
- **External Auxiliary Generalization:** `AIR_LEAK_AUX` files are evaluated strictly as an external test set and never mixed into IICA training.
- **Reproducible Artifacts:** All outputs are written to `output/` with complete model metadata JSON and serializable joblib bundle.


In [ ]:
# CELL 1 — IMPORT LIBRARIES
import os
import glob
import random
import json
import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Libraries imported successfully.")


In [ ]:
# CELL 2 — CONFIGURATION
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

SR = 16000
WINDOW_SEC = 2
WINDOW_SAMPLES = SR * WINDOW_SEC

NO_LEAK = 0
LEAK = 1
CLASS_NAMES = ["NO_LEAK", "LEAK"]

print("Sample rate:", SR)
print("Window:", WINDOW_SEC, "seconds")
print("Window samples:", WINDOW_SAMPLES)
print("Classes:", CLASS_NAMES)


In [ ]:
# CELL 3 — OUTPUT DIRECTORY CONFIGURATION
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_OUTPUT_DIR = OUTPUT_DIR / "model"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifact output directory:", OUTPUT_DIR.resolve())
print("Model output directory   :", MODEL_OUTPUT_DIR.resolve())


In [ ]:
# CELL 4 — DATASET PATHS
DATASETS = {
    "TUBELEAK": "/kaggle/input/datasets/jyotshanasr/tube-leak/tubeleak",
    "VENTLEAK": "/kaggle/input/datasets/jyotshanasr/vent-leak/ventleak",
    "VENTLOW": "/kaggle/input/datasets/jyotshanasr/vent-flow/ventlow"
}

ROOMTONE_PATH = "/kaggle/input/datasets/jyotshanasr/ductsense-no-leak-audio/Room_Record.wav"
MALL_PATH = "/kaggle/input/datasets/jyotshanasr/ductsense-no-leak-audio/Room_Record (2).wav"

AIR_LEAK_AUX_DIR = "/kaggle/input/datasets/jyotshanasr/air-leak-aux"

print("Dataset paths configured.")


In [ ]:
# CELL 5 — VERIFY DATASET PATHS
print("Checking IICA directories...\n")
for name, path in DATASETS.items():
    exists = os.path.exists(path)
    status_str = "FOUND" if exists else "NOT FOUND"
    print(f"{name:10s} -> {status_str}")

print("\nChecking NO_LEAK recordings...\n")
print("RoomTone2 ->", "FOUND" if os.path.exists(ROOMTONE_PATH) else "NOT FOUND")
print("Empty Mall ->", "FOUND" if os.path.exists(MALL_PATH) else "NOT FOUND")

missing_paths = [p for p in DATASETS.values() if not os.path.exists(p)]
if not os.path.exists(ROOMTONE_PATH):
    missing_paths.append(ROOMTONE_PATH)
if not os.path.exists(MALL_PATH):
    missing_paths.append(MALL_PATH)

if missing_paths:
    raise FileNotFoundError(
        "Kaggle dataset paths not accessible in current environment:\n"
        + "\n".join(f" - {p}" for p in missing_paths)
        + "\n\nPlease execute this notebook in Kaggle with the datasets attached."
    )

print("\nAll required files/directories found.")


In [ ]:
# CELL 6 — DISCOVER ALL IICA LEAK RECORDINGS
leak_records = []

for leak_class, root in DATASETS.items():
    wav_files = glob.glob(os.path.join(root, "**", "*.wav"), recursive=True)
    for path in wav_files:
        leak_records.append({
            "path": path,
            "source_class": leak_class,
            "label": LEAK
        })

leak_df = pd.DataFrame(leak_records)
print("Total IICA recordings:", len(leak_df))
print("\nRecordings by leak type:")
print(leak_df["source_class"].value_counts().sort_index())


In [ ]:
# CELL 7 — CHECK DUPLICATE RECORDING PATHS
duplicate_count = leak_df["path"].duplicated().sum()
print("Duplicate recording paths:", duplicate_count)
assert duplicate_count == 0, "Duplicate recording paths detected."
print("No duplicate recording paths.")


In [ ]:
# CELL 8 — RECORDING-LEVEL TRAIN / TEST SPLIT
train_leak_df, test_leak_df = train_test_split(
    leak_df,
    test_size=0.20,
    random_state=SEED,
    stratify=leak_df["source_class"]
)

train_leak_df = train_leak_df.reset_index(drop=True)
test_leak_df = test_leak_df.reset_index(drop=True)

print("Training leak recordings:", len(train_leak_df))
print("Testing leak recordings :", len(test_leak_df))
print("\nTRAIN distribution:")
print(train_leak_df["source_class"].value_counts().sort_index())
print("\nTEST distribution:")
print(test_leak_df["source_class"].value_counts().sort_index())


In [ ]:
# CELL 9 — LOAD NO_LEAK AUDIO
roomtone_audio, roomtone_sr = librosa.load(ROOMTONE_PATH, sr=SR, mono=True)
mall_audio, mall_sr = librosa.load(MALL_PATH, sr=SR, mono=True)

print("RoomTone2")
print("  Sample rate:", roomtone_sr)
print("  Duration:", len(roomtone_audio) / SR, "seconds")
print("  Samples:", len(roomtone_audio))

print("\nEmpty Mall")
print("  Sample rate:", mall_sr)
print("  Duration:", len(mall_audio) / SR, "seconds")
print("  Samples:", len(mall_audio))


In [ ]:
# CELL 10 — CREATE NON-OVERLAPPING NO_LEAK WINDOWS
def create_non_overlapping_windows(audio, window_samples=WINDOW_SAMPLES):
    windows = []
    number_of_windows = len(audio) // window_samples
    for i in range(number_of_windows):
        start = i * window_samples
        end = start + window_samples
        window = audio[start:end]
        if len(window) == window_samples:
            windows.append(window)
    return windows

roomtone_windows = create_non_overlapping_windows(roomtone_audio)
mall_windows = create_non_overlapping_windows(mall_audio)

print("RoomTone2 windows:", len(roomtone_windows))
print("Empty Mall windows:", len(mall_windows))


In [ ]:
# CELL 11 — TEMPORAL TRAIN / TEST SPLIT FOR NO_LEAK
def temporal_split(windows, train_fraction=0.80):
    split_index = int(len(windows) * train_fraction)
    return windows[:split_index], windows[split_index:]

room_train, room_test = temporal_split(roomtone_windows)
mall_train, mall_test = temporal_split(mall_windows)

print("RoomTone2:")
print("  Train:", len(room_train))
print("  Test :", len(room_test))

print("\nEmpty Mall:")
print("  Train:", len(mall_train))
print("  Test :", len(mall_test))


In [ ]:
# CELL 12 — COMBINE NO_LEAK WINDOWS
no_leak_train_audio = room_train + mall_train
no_leak_test_audio = room_test + mall_test

print("NO_LEAK training windows:", len(no_leak_train_audio))
print("NO_LEAK testing windows :", len(no_leak_test_audio))


In [ ]:
# CELL 13 — LIMIT NO_LEAK TRAINING SIZE
rng = np.random.default_rng(SEED)
MAX_NO_LEAK_TRAIN = 120

if len(no_leak_train_audio) > MAX_NO_LEAK_TRAIN:
    selected_indices = rng.choice(
        len(no_leak_train_audio),
        size=MAX_NO_LEAK_TRAIN,
        replace=False
    )
    no_leak_train_audio = [no_leak_train_audio[i] for i in selected_indices]

print("Final NO_LEAK training windows:", len(no_leak_train_audio))


In [ ]:
# CELL 14 — RANDOM WINDOW EXTRACTION FOR IICA
def extract_random_window(path):
    audio, _ = librosa.load(path, sr=SR, mono=True)
    if len(audio) < WINDOW_SAMPLES:
        audio = np.pad(audio, (0, WINDOW_SAMPLES - len(audio)))
        return audio[:WINDOW_SAMPLES]
    max_start = len(audio) - WINDOW_SAMPLES
    start = rng.integers(0, max_start + 1)
    return audio[start:start + WINDOW_SAMPLES]


In [ ]:
# CELL 15 — BASELINE FEATURE EXTRACTION (30 FEATURES)
def extract_features(audio):
    mfcc = librosa.feature.mfcc(y=audio, sr=SR, n_mfcc=13)
    mfcc_mean = np.mean(mfcc, axis=1)
    mfcc_std = np.std(mfcc, axis=1)

    centroid = librosa.feature.spectral_centroid(y=audio, sr=SR)
    centroid_mean = np.mean(centroid)
    centroid_std = np.std(centroid)

    bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=SR)
    bandwidth_mean = np.mean(bandwidth)
    bandwidth_std = np.std(bandwidth)

    return np.concatenate([
        mfcc_mean,
        mfcc_std,
        [centroid_mean, centroid_std],
        [bandwidth_mean, bandwidth_std]
    ])

test_feature = extract_features(no_leak_train_audio[0])
print("Number of extracted features:", len(test_feature))
assert len(test_feature) == 30


In [ ]:
# CELL 16 — BUILD LEAK TRAINING AUDIO
MAX_PER_LEAK_CLASS = 150
train_leak_selected = []

for leak_class in DATASETS.keys():
    candidates = train_leak_df[train_leak_df["source_class"] == leak_class]
    n_select = min(MAX_PER_LEAK_CLASS, len(candidates))
    selected = candidates.sample(n=n_select, random_state=SEED)
    train_leak_selected.append(selected)

train_leak_selected = pd.concat(train_leak_selected, ignore_index=True)
print("Selected LEAK training recordings:", len(train_leak_selected))
print(train_leak_selected["source_class"].value_counts().sort_index())


In [ ]:
# CELL 17 — BUILD LEAK TEST AUDIO
MAX_TEST_PER_LEAK_CLASS = 30
test_leak_selected = []

for leak_class in DATASETS.keys():
    candidates = test_leak_df[test_leak_df["source_class"] == leak_class]
    n_select = min(MAX_TEST_PER_LEAK_CLASS, len(candidates))
    selected = candidates.sample(n=n_select, random_state=SEED)
    test_leak_selected.append(selected)

test_leak_selected = pd.concat(test_leak_selected, ignore_index=True)
print("Selected LEAK test recordings:", len(test_leak_selected))
print(test_leak_selected["source_class"].value_counts().sort_index())


In [ ]:
# CELL 18 — CREATE TRAINING FEATURE MATRIX
X_train_list = []
y_train_list = []

for _, row in train_leak_selected.iterrows():
    audio = extract_random_window(row["path"])
    X_train_list.append(extract_features(audio))
    y_train_list.append(LEAK)

for audio in no_leak_train_audio:
    X_train_list.append(extract_features(audio))
    y_train_list.append(NO_LEAK)

X_train = np.asarray(X_train_list, dtype=np.float32)
y_train = np.asarray(y_train_list, dtype=np.int32)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("\nTraining class counts:")
print(pd.Series(y_train).map({0: "NO_LEAK", 1: "LEAK"}).value_counts())


In [ ]:
# CELL 19 — CREATE TEST FEATURE MATRIX WITH TRACEABILITY
X_test_list = []
y_test_list = []
test_source_list = []
test_leak_type_list = []
test_path_list = []

for _, row in test_leak_selected.iterrows():
    audio = extract_random_window(row["path"])
    X_test_list.append(extract_features(audio))
    y_test_list.append(LEAK)
    test_source_list.append("IICA")
    test_leak_type_list.append(row["source_class"])
    test_path_list.append(row["path"])

for i, audio in enumerate(room_test):
    X_test_list.append(extract_features(audio))
    y_test_list.append(NO_LEAK)
    test_source_list.append("RoomTone2")
    test_leak_type_list.append("NO_LEAK")
    test_path_list.append(f"{ROOMTONE_PATH}#win_{i}")

for i, audio in enumerate(mall_test):
    X_test_list.append(extract_features(audio))
    y_test_list.append(NO_LEAK)
    test_source_list.append("EmptyMall")
    test_leak_type_list.append("NO_LEAK")
    test_path_list.append(f"{MALL_PATH}#win_{i}")

X_test = np.asarray(X_test_list, dtype=np.float32)
y_test = np.asarray(y_test_list, dtype=np.int32)
test_source_list = np.asarray(test_source_list)
test_leak_type_list = np.asarray(test_leak_type_list)
test_path_list = np.asarray(test_path_list)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("\nTest class counts:")
print(pd.Series(y_test).map({0: "NO_LEAK", 1: "LEAK"}).value_counts())


In [ ]:
# CELL 20 — LEAKAGE OVERLAP CHECK
train_paths = set(train_leak_selected["path"])
test_paths = set(test_leak_selected["path"])
overlap = train_paths.intersection(test_paths)
print("IICA recording overlap:", len(overlap))
assert len(overlap) == 0, "ERROR: IICA recording overlap detected!"
print("PASS: No IICA recording appears in both train and test.")


In [ ]:
# CELL 21 — TRAIN RANDOM FOREST
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1
)
model.fit(X_train, y_train)
print("Random Forest training completed.")


In [ ]:
# CELL 22 — PREDICTIONS & PROBABILITIES
y_pred = model.predict(X_test)
leak_probability = model.predict_proba(X_test)[:, 1]
print("Predictions generated.")
print("Model classes:", model.classes_)
print("First 10 LEAK probabilities:", leak_probability[:10])


In [ ]:
# CELL 23 — SAVE TEST PREDICTIONS TO CSV
test_predictions_df = pd.DataFrame({
    "recording_path": test_path_list,
    "source": test_source_list,
    "leak_type": test_leak_type_list,
    "actual_label": y_test,
    "actual_class": np.where(y_test == LEAK, "LEAK", "NO_LEAK"),
    "predicted_label": y_pred,
    "predicted_class": np.where(y_pred == LEAK, "LEAK", "NO_LEAK"),
    "leak_probability": leak_probability
})

predictions_path = OUTPUT_DIR / "test_predictions.csv"
test_predictions_df.to_csv(predictions_path, index=False)
print("Saved test predictions to:", predictions_path)


In [ ]:
# CELL 24 — MAIN MODEL EVALUATION
accuracy = accuracy_score(y_test, y_pred)
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
leak_recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)

report_str = classification_report(
    y_test,
    y_pred,
    target_names=["NO_LEAK", "LEAK"],
    zero_division=0
)

print("=" * 65)
print("             DUCTSENSE BINARY RESULTS")
print("=" * 65)
print(f"Accuracy            : {accuracy:.4f}")
print(f"Balanced Accuracy   : {balanced_accuracy:.4f}")
print(f"Precision           : {precision:.4f}")
print(f"LEAK Recall         : {leak_recall:.4f}")
print(f"F1 Score            : {f1:.4f}")
print("\nClassification Report:\n")
print(report_str)


In [ ]:
# CELL 25 — CONFUSION MATRIX & SPECIFICITY
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

no_leak_specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
no_leak_fpr = fp / (tn + fp) if (tn + fp) > 0 else 0.0

print("Confusion Matrix:")
print(cm)
print("\nDetailed counts:")
print("True Negative  :", tn)
print("False Positive :", fp)
print("False Negative :", fn)
print("True Positive  :", tp)
print(f"\nNO_LEAK Specificity: {no_leak_specificity:.4f}")
print(f"NO_LEAK False Positive Rate: {no_leak_fpr:.4f}")


In [ ]:
# CELL 26 — SAVE METRICS FILE
metrics_path = OUTPUT_DIR / "metrics.txt"
with open(metrics_path, "w", encoding="utf-8") as f:
    f.write("DUCTSENSE BINARY AUDIO MODEL RESULTS\n")
    f.write("=" * 55 + "\n\n")
    f.write(f"Accuracy: {accuracy:.6f}\n")
    f.write(f"Balanced Accuracy: {balanced_accuracy:.6f}\n")
    f.write(f"Precision: {precision:.6f}\n")
    f.write(f"LEAK Recall: {leak_recall:.6f}\n")
    f.write(f"F1 Score: {f1:.6f}\n")
    f.write(f"NO_LEAK Specificity: {no_leak_specificity:.6f}\n")
    f.write(f"NO_LEAK False Positive Rate: {no_leak_fpr:.6f}\n\n")
    f.write("Confusion Matrix [TN FP; FN TP]\n")
    f.write(str(cm) + "\n\n")
    f.write(f"True Negative  : {tn}\n")
    f.write(f"False Positive : {fp}\n")
    f.write(f"False Negative : {fn}\n")
    f.write(f"True Positive  : {tp}\n\n")
    f.write("Classification Report:\n")
    f.write(report_str + "\n")

print("Saved metrics to:", metrics_path)


In [ ]:
# CELL 27 — CONFUSION MATRIX PLOT
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues", interpolation="nearest")
plt.title("DuctSense LEAK / NO_LEAK Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.xticks([0, 1], ["NO_LEAK", "LEAK"])
plt.yticks([0, 1], ["NO_LEAK", "LEAK"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="red" if i != j else "black")

plt.colorbar()
plt.tight_layout()

cm_path = OUTPUT_DIR / "confusion_matrix.png"
plt.savefig(cm_path, dpi=200, bbox_inches="tight")
print("Saved confusion matrix plot to:", cm_path)
plt.show()


In [ ]:
# CELL 28 — NO_LEAK PERFORMANCE BY RECORDING
print("=" * 65)
print("NO_LEAK PERFORMANCE BY RECORDING")
print("=" * 65)

environment_results = []
for source in ["RoomTone2", "EmptyMall"]:
    mask = test_source_list == source
    actual = y_test[mask]
    predicted = y_pred[mask]
    total = len(actual)

    no_leak_correct = int(np.sum(predicted == NO_LEAK))
    accuracy_source = no_leak_correct / total if total > 0 else 0.0
    false_leak_rate = np.sum(predicted == LEAK) / total if total > 0 else 0.0

    environment_results.append({
        "source": source,
        "samples": total,
        "correct_no_leak": no_leak_correct,
        "no_leak_accuracy": accuracy_source,
        "false_leak_rate": false_leak_rate
    })

    print(f"\n{source}")
    print("Samples:", total)
    print("Correctly classified NO_LEAK:", no_leak_correct)
    print(f"NO_LEAK accuracy: {accuracy_source:.4f}")
    print(f"False LEAK rate: {false_leak_rate:.4f}")

env_df = pd.DataFrame(environment_results)
env_path = OUTPUT_DIR / "no_leak_environment_results.csv"
env_df.to_csv(env_path, index=False)
print("\nSaved environment results to:", env_path)


In [ ]:
# CELL 29 — LEAK RECALL BY IICA LEAK TYPE
print("=" * 65)
print("LEAK PERFORMANCE BY IICA LEAK TYPE")
print("=" * 65)

leak_type_results = []
for leak_type in ["TUBELEAK", "VENTLEAK", "VENTLOW"]:
    mask = test_leak_type_list == leak_type
    predicted = y_pred[mask]
    total = int(np.sum(mask))
    detected = int(np.sum(predicted == LEAK))
    recall_type = detected / total if total > 0 else 0.0

    leak_type_results.append({
        "leak_type": leak_type,
        "detected": detected,
        "total": total,
        "recall": recall_type
    })

    print(f"{leak_type:10s} | Detected={detected:3d} / {total:3d} | Recall={recall_type:.4f}")

leak_type_df = pd.DataFrame(leak_type_results)
leak_path = OUTPUT_DIR / "leak_type_results.csv"
leak_type_df.to_csv(leak_path, index=False)
print("\nSaved leak type results to:", leak_path)


In [ ]:
# CELL 30 — LEAK PROBABILITY THRESHOLD ANALYSIS
thresholds = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
threshold_results = []

for threshold in thresholds:
    t_pred = (leak_probability >= threshold).astype(int)
    acc_t = accuracy_score(y_test, t_pred)
    bacc_t = balanced_accuracy_score(y_test, t_pred)
    prec_t = precision_score(y_test, t_pred, zero_division=0)
    rec_t = recall_score(y_test, t_pred, zero_division=0)
    f1_t = f1_score(y_test, t_pred, zero_division=0)

    no_leak_mask = (y_test == NO_LEAK)
    if np.sum(no_leak_mask) > 0:
        fpr_t = np.sum(t_pred[no_leak_mask] == LEAK) / np.sum(no_leak_mask)
        spec_t = np.sum(t_pred[no_leak_mask] == NO_LEAK) / np.sum(no_leak_mask)
    else:
        fpr_t = 0.0
        spec_t = 0.0

    threshold_results.append({
        "threshold": threshold,
        "accuracy": acc_t,
        "balanced_accuracy": bacc_t,
        "precision": prec_t,
        "leak_recall": rec_t,
        "f1": f1_t,
        "no_leak_false_positive_rate": fpr_t,
        "no_leak_specificity": spec_t
    })

threshold_df = pd.DataFrame(threshold_results)
print(threshold_df.to_string(index=False))

threshold_path = OUTPUT_DIR / "threshold_results.csv"
threshold_df.to_csv(threshold_path, index=False)
print("Saved threshold analysis:", threshold_path)


In [ ]:
# CELL 31 — RANDOM FOREST FEATURE IMPORTANCE
feature_names = (
    [f"MFCC_{i+1}_mean" for i in range(13)]
    + [f"MFCC_{i+1}_std" for i in range(13)]
    + ["Centroid_mean", "Centroid_std", "Bandwidth_mean", "Bandwidth_std"]
)
assert len(feature_names) == 30

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False).reset_index(drop=True)

print(feature_importance_df.head(15).to_string(index=False))

fi_path = OUTPUT_DIR / "feature_importance.csv"
feature_importance_df.to_csv(fi_path, index=False)
print("Saved feature importance:", fi_path)

print("\nNOTE: Feature importance reflects statistical association in this dataset;")
print("it does not define a physical leak frequency range.")


In [ ]:
# CELL 32 — SAVE TRAINED MODEL + COMPLETE METADATA
MODEL_PATH = MODEL_OUTPUT_DIR / "ductsense_binary_rf_v2.pkl"
METADATA_PATH = MODEL_OUTPUT_DIR / "ductsense_binary_rf_v2_meta.json"

EXPORT_THRESHOLD = 0.50

model_package = {
    "model": model,
    "sample_rate": SR,
    "window_seconds": WINDOW_SEC,
    "window_samples": WINDOW_SAMPLES,
    "feature_names": feature_names,
    "classes": CLASS_NAMES,
    "threshold": EXPORT_THRESHOLD
}
joblib.dump(model_package, MODEL_PATH)

metadata = {
    "model_type": "RandomForestClassifier",
    "model_version": "ductsense_binary_rf_v2",
    "export_timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "sample_rate": int(SR),
    "window_seconds": int(WINDOW_SEC),
    "window_samples": int(WINDOW_SAMPLES),
    "feature_count": len(feature_names),
    "feature_names": feature_names,
    "class_mapping": {"0": "NO_LEAK", "1": "LEAK"},
    "random_seed": int(SEED),
    "threshold": EXPORT_THRESHOLD,
    "training_strategy": "Recording-level 80/20 stratified split for IICA leaks; non-overlapping temporal split (80/20, max 120 train windows) for NO_LEAK recordings.",
    "dataset_sources": {
        "TUBELEAK": DATASETS["TUBELEAK"],
        "VENTLEAK": DATASETS["VENTLEAK"],
        "VENTLOW": DATASETS["VENTLOW"],
        "ROOMTONE": ROOMTONE_PATH,
        "EMPTY_MALL": MALL_PATH
    },
    "rf_parameters": {
        "n_estimators": 300,
        "max_depth": 12,
        "class_weight": "balanced",
        "random_state": SEED,
        "n_jobs": -1
    },
    "evaluation_metrics": {
        "accuracy": float(accuracy),
        "balanced_accuracy": float(balanced_accuracy),
        "precision": float(precision),
        "leak_recall": float(leak_recall),
        "f1": float(f1),
        "no_leak_specificity": float(no_leak_specificity),
        "no_leak_false_positive_rate": float(no_leak_fpr),
        "confusion_matrix": {
            "true_negative": int(tn),
            "false_positive": int(fp),
            "false_negative": int(fn),
            "true_positive": int(tp)
        }
    }
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Model saved successfully:", MODEL_PATH)
print("Metadata saved successfully:", METADATA_PATH)


In [ ]:
# CELL 33 — SINGLE 2-SECOND INFERENCE FUNCTION
def predict_audio_file(audio_path, threshold=0.50):
    audio, _ = librosa.load(audio_path, sr=SR, mono=True)
    if len(audio) < WINDOW_SAMPLES:
        audio = np.pad(audio, (0, WINDOW_SAMPLES - len(audio)))
    else:
        audio = audio[:WINDOW_SAMPLES]

    features = extract_features(audio).reshape(1, -1)
    probability = float(model.predict_proba(features)[0, 1])
    prediction = "LEAK" if probability >= threshold else "NO_LEAK"
    return prediction, probability

pred_room, prob_room = predict_audio_file(ROOMTONE_PATH)
print("RoomTone2 -> Prediction:", pred_room, f"| LEAK probability: {prob_room:.4f}")

pred_mall, prob_mall = predict_audio_file(MALL_PATH)
print("Empty Mall -> Prediction:", pred_mall, f"| LEAK probability: {prob_mall:.4f}")


In [ ]:
# CELL 34 — EXTERNAL AIR_LEAK_AUX GENERALIZATION TEST
print("=" * 65)
print("EXTERNAL AIR_LEAK_AUX GENERALIZATION TEST")
print("=" * 65)

aux_files = glob.glob(os.path.join(AIR_LEAK_AUX_DIR, "*.wav")) if os.path.exists(AIR_LEAK_AUX_DIR) else []
if aux_files:
    print(f"Found {len(aux_files)} external auxiliary recordings:")
    aux_results = []
    for af in sorted(aux_files):
        p, prob = predict_audio_file(af, threshold=EXPORT_THRESHOLD)
        aux_results.append({
            "file": os.path.basename(af),
            "predicted_class": p,
            "leak_probability": prob
        })
        print(f"  {os.path.basename(af):20s} -> Predicted: {p:7s} | LEAK Prob: {prob:.4f}")

    aux_df = pd.DataFrame(aux_results)
    aux_path = OUTPUT_DIR / "external_air_leak_aux_predictions.csv"
    aux_df.to_csv(aux_path, index=False)
    print("\nSaved external auxiliary predictions to:", aux_path)
    print("\nCAVEAT: These 5 external recordings represent an exploratory sanity test,")
    print("NOT definitive proof of acoustic domain generalization.")
else:
    print(f"AIR_LEAK_AUX directory not found at: {AIR_LEAK_AUX_DIR}")
    print("Skipping external auxiliary generalization test.")


In [ ]:
# CELL 35 — VERIFY MODEL ARTIFACT RELOAD
print("=" * 65)
print("VERIFYING MODEL ARTIFACT RELOAD")
print("=" * 65)

assert MODEL_PATH.exists(), f"Model file missing: {MODEL_PATH}"
assert METADATA_PATH.exists(), f"Metadata file missing: {METADATA_PATH}"

reloaded_pkg = joblib.load(MODEL_PATH)
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    reloaded_meta = json.load(f)

assert "model" in reloaded_pkg
assert reloaded_pkg["sample_rate"] == SR
assert reloaded_pkg["threshold"] == EXPORT_THRESHOLD
assert len(reloaded_meta["feature_names"]) == 30

test_sample = no_leak_train_audio[0]
sample_features = extract_features(test_sample).reshape(1, -1)
reloaded_prob = float(reloaded_pkg["model"].predict_proba(sample_features)[0, 1])

print("Reloaded model verification passed.")
print(f"Sample test probability: {reloaded_prob:.4f}")


In [ ]:
# CELL 36 — FINAL ARTIFACT AUDIT
print("=" * 65)
print("FINAL ARTIFACT AUDIT")
print("=" * 65)

expected_files = [
    OUTPUT_DIR / "metrics.txt",
    OUTPUT_DIR / "confusion_matrix.png",
    OUTPUT_DIR / "threshold_results.csv",
    OUTPUT_DIR / "feature_importance.csv",
    OUTPUT_DIR / "test_predictions.csv",
    OUTPUT_DIR / "no_leak_environment_results.csv",
    OUTPUT_DIR / "leak_type_results.csv",
    MODEL_OUTPUT_DIR / "ductsense_binary_rf_v2.pkl",
    MODEL_OUTPUT_DIR / "ductsense_binary_rf_v2_meta.json",
]

all_passed = True
for fpath in expected_files:
    exists = fpath.exists()
    size_kb = (fpath.stat().st_size / 1024) if exists else 0
    status = "EXISTS" if exists else "MISSING"
    print(f"{status:8s} | {size_kb:8.2f} KB | {fpath}")
    if not exists:
        all_passed = False

if all_passed:
    print("\nALL EXPECTED OUTPUT ARTIFACTS VERIFIED.")
else:
    print("\nWARNING: Some expected artifacts are missing.")
